# Phase 3 — BAML LC6 Extraction Walkthrough

Interactive walkthrough of the LC6 BAML extraction pipeline.
Per the multi-stage plan (see AGENTS.md).

This notebook picks a single subject, reads every Markdown under
`data/bi_ep/syllabi_md/`, runs all 5 LC6 BAML functions concurrently,
and shows the resulting rows in the dev SQLite table.

When `baml_client` is importable, the BAML functions are called
directly. When it's missing (the dev path), a deterministic stub
with the same shape is used.

In [ ]:
# 1. Pick the subject + language + check what's available.
import pathlib, sys

MD_ROOT = pathlib.Path("data/bi_ep/syllabi_md")
SQLITE = pathlib.Path("data/bi_ep/extracted_syllabi.sqlite")
SUBJECT = "mathematics"  # one of: mathematics, english, gaeilge, chemistry, biology, physics, geography, computer_science
LANGUAGE = "en"

print(f"MD root exists: {MD_ROOT.exists()}")
print(f"SQLite path: {SQLITE}")
if MD_ROOT.exists():
    mds = sorted(MD_ROOT.rglob("*.md"))
    print(f"Available .md files: {len(mds)}")
    for p in mds[:5]:
        print(f"  {p.relative_to(MD_ROOT.parent.parent.parent):80s} {p.stat().st_size / 1024:8.1f} KiB")
else:
    print("No .md files yet. Run `python -m cocoindex_flows.pdf.pdf_to_markdown_app` first.")

In [ ]:
# 2. Run the extraction App.
from cocoindex_flows.education.lc6_extraction_app import run

stats = run(subject_slug=SUBJECT, language=LANGUAGE)
print(stats)

In [ ]:
# 3. Inspect the resulting rows.
import sqlite3, json

if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        rows = conn.execute(
            "SELECT subnation, subject_slug, language, "
            "substr(syllabus_json, 1, 120), fetched_at "
            "FROM extracted_syllabi "
            "WHERE subject_slug = ? AND language = ?",
            (SUBJECT, LANGUAGE)
        ).fetchall()
    print(f"Rows for {SUBJECT} ({LANGUAGE}): {len(rows)}")
    for r in rows:
        print(f"  {r[0]:20s} {r[1]:20s} {r[2]:3s}  {r[3]}...")

    # Inspect one row's full JSON shape
    sample = conn.execute(
        "SELECT syllabus_json, exam_paper_json, marking_json, "
        "concepts_json, diagrams_json FROM extracted_syllabi LIMIT 1"
    ).fetchone()
    for i, key in enumerate(["syllabus", "exam_paper", "marking", "concepts", "diagrams"]):
        print(f"\n--- {key}_json ---")
        if sample[i]:
            parsed = json.loads(sample[i])
            print(f"  keys: {list(parsed.keys())}")
            print(f"  subject_slug: {parsed.get('subject_slug')}")
            print(f"  stub: {parsed.get('stub', False)}")
        else:
            print("  (empty)")
else:
    print("No SQLite database yet.")

## Summary

- Every .md under `data/bi_ep/syllabi_md/` becomes one row in `extracted_syllabi` (composite primary key on `(subnation, stage, subject_slug, language, source_pdf)`).
- Each row has 5 JSON fields (`syllabus`, `exam_paper`, `marking`, `concepts`, `diagrams`) — one per LC6 BAML function.
- When `baml_client` is importable, the BAML functions are called via `asyncio.gather`. When missing, a stub with `stub: True` is used so the pipeline stays runnable.
- Ready for Phase 4 (equivalency graph) which reads the `concepts_json` field to build the cross-jurisdiction topic map.